# Comparativo de estrategias de chunking

**LAB-26 · F1-09** · Este notebook responde a una pregunta concreta de nuestro RAG: *¿cambia lo que recupera el
asistente según cómo se corten los documentos en fragmentos (chunks)?*

Se comparan **tres estrategias** sobre el mismo corpus (`data/kb`, 24 políticas y FAQs de ShopLab en español):

| Estrategia | Cómo corta |
|---|---|
| **Tamaño fijo** | cada `chunk_size` caracteres, con solape; no mira el contenido |
| **Por encabezados** | por secciones del Markdown; es el troceado de producción de `ingest/` (ADR-0016) |
| **Semántica** | donde cambia el significado entre frases consecutivas, medido con embeddings |

Después se lanzan **20 preguntas de prueba** contra los chunks de cada estrategia y se mide si el chunk correcto
aparece entre los `top_k` recuperados.

**Cómo leerlo:** las celdas se ejecutan de arriba abajo. La celda 1 declara todos los parámetros; la 2 comprueba que
Ollama y el modelo de embeddings están disponibles (si no, se detiene antes de calcular nada). Este notebook **mide y
concluye**, pero no cambia el chunking de producción.

In [1]:
from chunking_lab.config import ExperimentConfig

config = ExperimentConfig(
    chunk_size=1000,           # tamaño máximo de un chunk en caracteres; igual en las tres estrategias
    chunk_overlap=100,         # caracteres repetidos entre chunks consecutivos (tamaño fijo y encabezados)
    top_k=4,                   # chunks recuperados por pregunta; el chat de producción usa 4
    breakpoint_percentile=80,  # semántica: corta donde la distancia supera este percentil del documento
    min_chunk_chars=200,       # semántica: los fragmentos más cortos se unen al vecino más cercano
)
# El modelo de embeddings y la URL salen de OLLAMA_EMBED_MODEL y OLLAMA_BASE_URL (bge-m3 y localhost:11434).
print(f"modelo={config.embed_model} · chunk_size={config.chunk_size} · chunk_overlap={config.chunk_overlap}"
      f" · top_k={config.top_k} · percentil={config.breakpoint_percentile} · min_chunk_chars={config.min_chunk_chars}")

modelo=bge-m3 · chunk_size=1000 · chunk_overlap=100 · top_k=4 · percentil=80 · min_chunk_chars=200


In [2]:
from chunking_lab.embeddings import OllamaEmbedder

embedder = OllamaEmbedder(config)
model_digest = embedder.preflight()  # PreflightError, con instrucciones, si Ollama o el modelo no están
print(f"Ollama disponible · modelo {config.embed_model} · digest {model_digest[:12] or 'no informado'}")

Ollama disponible · modelo bge-m3 · digest 790764642607


## 1. El corpus

Se usan los documentos Markdown de la base de conocimiento, sin modificarlos. Los PDF se excluyen porque repiten el
texto de su Markdown y aquí no se compara el parseo de PDF.

In [3]:
import pandas as pd

from chunking_lab.corpus import load_corpus

documents = load_corpus(config.kb_dir)
corpus_table = (
    pd.DataFrame({"tema": [d.topic for d in documents], "caracteres": [len(d.body) for d in documents]})
    .groupby("tema").agg(documentos=("tema", "size"), caracteres=("caracteres", "sum"))
)
corpus_table.loc["total"] = corpus_table.sum()
print(f"{len(documents)} documentos; media de {sum(len(d.body) for d in documents) // len(documents)} caracteres")
corpus_table

24 documentos; media de 1442 caracteres


,documentos,caracteres
tema,,
payments,5,6923
privacy,5,7056
returns,5,7839
shipping,5,7084
warranty,4,5727
total,24,34629


## 2. Las tres estrategias

Las tres se aplican con el mismo `chunk_size`. Cada una devuelve chunks con su documento y su posición.

* **Tamaño fijo** corta en el carácter número `chunk_size`, aunque sea a mitad de palabra, de frase o de tabla.
* **Por encabezados** nunca mezcla dos secciones y conserva la ruta de encabezados de cada chunk; si una sección es
  más larga que `chunk_size`, la divide en párrafos, líneas y frases.
* **Semántica** parte el texto en unidades (frases, elementos de lista, tablas), calcula con `bge-m3` la distancia entre
  unidades consecutivas y corta donde esa distancia supera el percentil `breakpoint_percentile` del propio documento.
  Si un grupo excede `chunk_size` lo vuelve a partir en su mayor distancia interna, y si es menor que `min_chunk_chars`
  lo une al vecino más cercano. No usa solape.

In [4]:
from chunking_lab.strategies import Strategy, build_chunks

chunks = {strategy: build_chunks(strategy, documents, config, embedder) for strategy in Strategy}
{strategy.value: len(found) for strategy, found in chunks.items()}

{'fixed': 48, 'headings': 138, 'semantic': 59}

### Cómo decide cortar la estrategia semántica

La curva muestra, para un documento, la distancia entre cada unidad y la siguiente. Los puntos rojos por encima del
umbral son los cortes: ahí el significado cambia más que en el resto del documento.

In [5]:
from chunking_lab.report import plot_distance_curve
from chunking_lab.strategies.semantic import adjacent_distances, breakpoint_threshold, semantic_units

example_document = next(d for d in documents if d.id == "returns-01-return-policy")
units = semantic_units(example_document.body, config.chunk_size)
distances = adjacent_distances(units, embedder)
threshold = breakpoint_threshold(distances, config.breakpoint_percentile)
plot_distance_curve(distances, threshold, f"{example_document.id}: {len(units)} unidades")

<Figure size 800x280 with 1 Axes>

## 3. Cuántos chunks produce cada estrategia

Con documentos de pocos miles de caracteres, el tamaño máximo por defecto (1 000) hace que el **tamaño fijo** produzca
pocos chunks casi del tamaño de un documento, mientras que **encabezados** produce muchos chunks pequeños. Esta
diferencia de granularidad forma parte de lo que se compara.

In [6]:
import pandas as pd

from chunking_lab.report import STRATEGY_LABELS, plot_size_histograms, size_summary

sizes = size_summary(chunks)
size_table = pd.DataFrame(sizes).T
size_table.index = [STRATEGY_LABELS[s] for s in size_table.index]
size_table.columns = ["chunks", "mínimo", "media", "máximo"]
size_table = size_table.astype({"chunks": int, "mínimo": int, "máximo": int})
display(size_table)
plot_size_histograms(chunks, config.chunk_size)

,chunks,mínimo,media,máximo
Tamaño fijo,48,225,771.4,1000
Por encabezados,138,72,242.3,731
Semántica,59,191,582.4,997


<Figure size 1200x300 with 3 Axes>

## 4. Un mismo pasaje, tres cortes

Un pasaje cuya respuesta cae cerca de un límite de chunk. Abajo se ve, para cada estrategia, qué chunk(s) contienen su
inicio y si el pasaje completo quedó **partido** (ningún chunk lo contiene entero).

In [7]:
from chunking_lab.report import cut_example
from chunking_lab.text import normalize

example_id = "payments-02-installments"
example_passage = (
    "El número de cuotas disponible depende del total del pedido, porque cada cuota debe ser de al menos R$ 30,00."
)
for strategy, example in cut_example(chunks, example_id, example_passage).items():
    print(f"=== {STRATEGY_LABELS[strategy]} · {'PASAJE PARTIDO' if example.cut else 'pasaje entero en un chunk'} ===")
    for found in example.chunks:
        text = normalize(found.text)
        start = text.find(normalize(example_passage)[:20])
        print(f"[chunk {found.position}, {len(found.text)} caracteres]")
        print("  ..." + text[max(0, start - 60):start + len(example_passage) + 40] + "...")
    print()

=== Tamaño fijo · PASAJE PARTIDO ===
[chunk 0, 1000 caracteres]
  ...nterés; la de 4 cuotas tiene interés. ## Casos especiales - El número de cuotas disponible depende del total del pedido, porque cada cuota debe ser de al menos R$ 30,0...

=== Por encabezados · pasaje entero en un chunk ===
[chunk 3, 297 caracteres]
  ...Casos especiales - El número de cuotas disponible depende del total del pedido, porque cada cuota debe ser de al menos R$ 30,00. - Si devuelves el pedido, el reembolso ...

=== Semántica · pasaje entero en un chunk ===
[chunk 0, 995 caracteres]
  ...nterés; la de 4 cuotas tiene interés. ## Casos especiales - El número de cuotas disponible depende del total del pedido, porque cada cuota debe ser de al menos R$ 30,00....



## 5. Las 20 preguntas de prueba

Son veinte preguntas en español, **cuatro por cada tema**, escritas como las haría un cliente. Cada una lleva el
documento donde está la respuesta y el **pasaje de referencia**, el texto literal que la contiene. Se guardan como datos
(`notebooks/data/questions.toml`), separados de este código.

Antes de medir nada se validan: 20 preguntas, 4 por tema, ids únicos, que el pasaje exista en su documento y **solo** en
ese, y que mida entre 40 y 300 caracteres. Si alguna regla falla, la celda se detiene y las lista todas.

In [8]:
from chunking_lab.questions import load_questions, validate_questions

questions = load_questions(config.questions_path)
validate_questions(questions, documents)
pd.DataFrame([{"id": q.id, "tema": q.topic, "pregunta": q.question, "documento esperado": q.document}
              for q in questions]).set_index("id")

,tema,pregunta,documento esperado
id,,,
Q01,payments,"Si mi compra es de R$ 120, ¿puedo pagarla en 5...",payments-02-installments
Q02,payments,"Mi boleto venció sin que lo pagara, ¿qué puedo...",payments-03-boleto-and-vouchers
Q03,payments,Ya intenté pagar tres veces el mismo pedido y ...,payments-04-payment-issues
Q04,payments,"Veo un cobro de ShopLab que no reconozco, ¿qué...",payments-04-payment-issues
Q05,privacy,¿Cómo borro las cookies que el sitio ya guardó...,privacy-04-cookies
Q06,privacy,¿Puedo darle mi contraseña al asistente para q...,privacy-02-data-collection-and-use
Q07,privacy,¿Qué información debo incluir en el correo par...,privacy-03-data-rights
Q08,privacy,¿Con quién comparte ShopLab mis datos personales?,privacy-01-privacy-policy
Q09,returns,¿Qué tengo que indicar al pedir la devolución ...,returns-02-return-process


## 6. Recuperación y tabla de resultados

Para cada estrategia se calcula el embedding de todos sus chunks y de cada pregunta (el mismo modelo en las tres), y se
recuperan los `top_k` chunks más cercanos por coseno **entre todos los documentos del corpus**, como hace el chat.

Un chunk recuperado cuenta como **correcto solo si procede del documento esperado y contiene el pasaje de referencia**;
un chunk del documento correcto que no incluye el pasaje no cuenta. Para cada pregunta se anota la **posición** (de 1 a
`top_k`) del primer chunk correcto, o «no encontrado». Con eso:

* **hit rate@k**: fracción de preguntas con algún chunk correcto entre los `top_k`.
* **MRR**: media de `1 / posición` (0 si no se encontró); premia que el correcto salga primero.

In [9]:
from chunking_lab.retrieval import ChunkIndex, run_retrieval

retrievals = {
    strategy: run_retrieval(ChunkIndex.build(chunks[strategy], embedder), questions, embedder, config.top_k)
    for strategy in Strategy
}
ranks = {strategy: {r.question_id: r.rank for r in found} for strategy, found in retrievals.items()}
{STRATEGY_LABELS[s]: sum(1 for r in found.values() if r is not None) for s, found in ranks.items()}

{'Tamaño fijo': 18, 'Por encabezados': 17, 'Semántica': 17}

In [10]:
from datetime import UTC, datetime

from chunking_lab.report import build_results, plot_metrics, results_table, validate_results

results = build_results(config, sizes, ranks, questions, model_digest, len(documents),
                        datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"))
validate_results(results)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(results_table(results).set_index("pregunta"))
plot_metrics(results)

,tema,documento esperado,Tamaño fijo,Por encabezados,Semántica
pregunta,,,,,
Q01,payments,payments-02-installments,1,2,1
Q02,payments,payments-03-boleto-and-vouchers,1,3,1
Q03,payments,payments-04-payment-issues,no encontrado,no encontrado,no encontrado
Q04,payments,payments-04-payment-issues,1,1,1
Q05,privacy,privacy-04-cookies,1,1,2
Q06,privacy,privacy-02-data-collection-and-use,1,1,1
Q07,privacy,privacy-03-data-rights,1,2,1
Q08,privacy,privacy-01-privacy-policy,1,2,2
Q09,returns,returns-02-return-process,1,4,1


<Figure size 600x300 with 1 Axes>

## 7. Dónde difieren las estrategias

La tabla dice *cuál* acertó; aquí se mira *por qué*. Se listan las preguntas en las que las estrategias no obtienen la
misma posición (primero las de mayor diferencia) y se abren tres de ellas: qué recuperó cada estrategia, en qué orden,
con su puntuación de coseno, y cuál era el chunk correcto (✔).

In [11]:
from chunking_lab.report import differing_cases, headline, retrieval_rows

cases = differing_cases(results)
print(f"{len(cases)} de {len(questions)} preguntas tienen una posición distinta según la estrategia:")
print(", ".join(case["id"] for case in cases))

14 de 20 preguntas tienen una posición distinta según la estrategia:
Q14, Q19, Q09, Q10, Q02, Q20, Q01, Q05, Q07, Q08, Q11, Q12, Q16, Q17


In [12]:
questions_by_id = {q.id: q for q in questions}


def show_case(question_id, width=100):
    question = questions_by_id[question_id]
    print(f"{question.id} · {question.question}")
    print(f"   respuesta esperada en {question.document}: «{question.passage[:width]}…»")
    for strategy in Strategy:
        retrieval = next(r for r in retrievals[strategy] if r.question_id == question_id)
        print(f"\n  {STRATEGY_LABELS[strategy]} → posición {retrieval.rank or 'no encontrado'}")
        for row in retrieval_rows(retrieval, question):
            mark = "✔" if row["correct"] else " "
            print(f"   {mark} {row['place']}. {row['score']:.3f} {row['document']} #{row['position']} "
                  f"({len(row['text'])} car.) {normalize(row['text'])[:width]}…")

### Caso 1 · Q14: lo acierta solo la estrategia por encabezados

In [13]:
show_case("Q14")

Q14 · Compré productos de varios vendedores, ¿llegará todo junto?
   respuesta esperada en shipping-01-delivery-times: «Los pedidos con productos de varios vendedores pueden llegar en envíos separados, cada uno con su pr…»

  Tamaño fijo → posición no encontrado
     1. 0.529 shipping-04-delivery-issues #0 (1000 car.) # Problemas con la entrega ## Alcance Este documento cubre los tres problemas más habituales de una …
     2. 0.491 shipping-04-delivery-issues #1 (685 car.) vered` pero no lo recibiste, abre un caso igualmente: se tratará como extravío. - Un producto dañado…
     3. 0.490 payments-04-payment-issues #0 (1000 car.) # Problemas con el pago ## Alcance Este documento cubre los problemas más habituales al pagar: pago …
     4. 0.487 shipping-01-delivery-times #0 (1000 car.) # Plazos de entrega ## Alcance Este documento explica cuánto tarda un pedido de ShopLab desde que se…

  Por encabezados → posición 1
   ✔ 1. 0.539 shipping-01-delivery-times #3 (339 car.) Casos especiales 

**Q14** (*varios vendedores, ¿llega todo junto?*). La respuesta es una viñeta de la sección «Casos especiales» al final de
`shipping-01-delivery-times`. Con **encabezados** esa sección es un chunk pequeño que se queda con el primer puesto. Con
**tamaño fijo** y **semántica** la viñeta cae dentro de un chunk grande que arranca con el título y el alcance del
documento; el que llegó al top 4 es el primero (que habla de plazos en general), no el que contiene la viñeta. Se acertó
el documento, pero no el fragmento: el chunk correcto quedó diluido entre otro contenido.

### Caso 2 · Q19: lo falla solo la estrategia por encabezados

In [14]:
show_case("Q19")

Q19 · ¿Qué debo preparar antes de reclamar la garantía de un producto?
   respuesta esperada en warranty-03-warranty-claims: «Reúne el número de pedido y una descripción del defecto, con fotos o un vídeo si es posible.…»

  Tamaño fijo → posición 1
   ✔ 1. 0.691 warranty-03-warranty-claims #0 (1000 car.) # Cómo reclamar la garantía ## Alcance Este documento describe cómo presentar una reclamación por un…
     2. 0.638 warranty-01-legal-warranty #0 (1000 car.) # Garantía legal ## Alcance Todos los productos vendidos por ShopLab tienen garantía legal contra de…
     3. 0.625 warranty-04-faq-warranty #0 (1000 car.) # Preguntas frecuentes sobre garantías ## Alcance Respuestas rápidas sobre garantías. El detalle est…
     4. 0.612 warranty-02-extended-warranty #1 (540 car.) plo: para un producto de R$ 500,00, la garantía extendida cuesta R$ 50,00. ## Casos especiales - No …

  Por encabezados → posición no encontrado
     1. 0.646 warranty-01-legal-warranty #1 (341 car.) Cómo funciona La ga

**Q19** (*¿qué preparar antes de reclamar la garantía?*). Aquí ocurre lo contrario. Con **tamaño fijo** y **semántica**, el
chunk que abre `warranty-03-warranty-claims` contiene el título, el alcance y los primeros pasos, y es el más parecido a la
pregunta. Con **encabezados** el pasaje vive en un chunk que empieza con el título genérico «Cómo funciona», el mismo
que tienen los 24 documentos, y no incluye el nombre del documento; en su lugar entran otros chunks sobre garantías (de
la garantía legal, de la FAQ y de la extendida) que se parecen más a la pregunta. Un chunk pequeño pierde el contexto
que lo identifica.

### Caso 3 · Q03: ninguna estrategia lo resuelve (y por qué el fallo no es todo culpa del chunking)

In [15]:
show_case("Q03")

Q03 · Ya intenté pagar tres veces el mismo pedido y no pasa, ¿qué hago?
   respuesta esperada en payments-04-payment-issues: «Si superas los 3 intentos en 24 horas, espera o elige otro método de pago.…»

  Tamaño fijo → posición no encontrado
     1. 0.645 payments-04-payment-issues #0 (1000 car.) # Problemas con el pago ## Alcance Este documento cubre los problemas más habituales al pagar: pago …
     2. 0.582 payments-05-faq-payments #1 (507 car.) a 3 intentos en 24 horas con el mismo pedido. ## ¿Qué hago si me cobraron dos veces? Escribe a sopor…
     3. 0.561 payments-01-payment-methods #1 (532 car.) único, sin cuotas | Mientras el pago no se apruebe, el pedido permanece en `created`; al aprobarse p…
     4. 0.554 payments-05-faq-payments #0 (1000 car.) # Preguntas frecuentes sobre pagos ## Alcance Respuestas rápidas sobre pagos. El detalle está en los…

  Por encabezados → posición no encontrado
     1. 0.716 payments-05-faq-payments #5 (138 car.) ¿Qué hago si mi pago es rechazado

**Q03** (*tres intentos de pago sin éxito*). Falla en las tres. Con **encabezados** el primer chunk recuperado es un
fragmento de la FAQ de pagos, `payments-05-faq-payments`, que *también* menciona los tres intentos en 24 horas. Contesta
razonablemente la pregunta, pero la medición solo da por buena la frase de `payments-04-payment-issues`, así que cuenta
como fallo. Es un límite del método (el corpus repite hechos entre políticas y FAQs aunque el texto literal sea
distinto), no solo de la estrategia.

## 8. Conclusiones y límites

In [16]:
print(headline(results))

Con las 20 preguntas de prueba: en MRR gana Tamaño fijo (0.817), 0.054 por encima de la siguiente; en hit rate@4 gana Tamaño fijo (90.0%), 5.0 puntos por encima de la siguiente.


**Lectura** (escrita sobre la ejecución guardada; si se cambian los parámetros o el corpus, hay que releerla):

* **Con este corpus, tamaño fijo y semántica recuperan mejor que encabezados en MRR**, y la diferencia se concentra en la
  *posición*: los chunks grandes llevan el título y el alcance del documento, así que la pregunta llega antes al
  documento correcto (Q09, Q19). Son documentos cortos (de unos 1 400 caracteres de media): un chunk de tamaño fijo es
  casi un documento entero, y recuperar un documento entero es lo que más premia esta medición.
* **Encabezados** produce muchos chunks pequeños y precisos (Q14 es la prueba), pero cuyo texto empieza con títulos
  genéricos como «Cómo funciona» o «Casos especiales», repetidos en todos los documentos, y sin el nombre del documento.
  Ese contexto perdido es la causa más probable de que a veces no entren en el top 4 (Q19, Q20).
* **Tamaño fijo** acierta más preguntas, pero cuando la respuesta queda en una ventana posterior, la ventana
  que la contiene no siempre llega al top 4 (Q14, Q03). La **semántica** queda entre las dos: sus chunks también arrancan
  con el título del documento y respetan mejor las frases, pero sufre las mismas diluciones (Q14, Q20).
* **El tamaño del chunk es parte de lo que se compara**: no se midió cuánto ruido recibe el modelo de lenguaje con un chunk
  grande frente a uno pequeño, que es lo que justifica trocear por secciones.

**Límites de esta comparación**

* **Veinte preguntas son una muestra pequeña**: un solo acierto cambia el hit rate 5 puntos. Las diferencias entre
  estrategias no son estadísticamente concluyentes.
* **El corpus es sintético, en español y de documentos cortos**; con documentos largos las conclusiones podrían cambiar.
* **La estrategia semántica depende de su percentil** (`breakpoint_percentile`) y de `min_chunk_chars`; aquí se probó un
  solo valor.
* **El acierto exige el documento esperado**: si una FAQ contesta igual de bien, cuenta como fallo (Q03). Las preguntas se
  eligieron con hechos poco repetidos para minimizarlo, pero no se elimina.
* **No se aplicó el umbral de similitud del chat** (0,6) ni se midió la calidad de la respuesta generada (fidelidad,
  relevancia); solo se compara qué chunks se recuperan.
* **No cambia el chunking de producción.** Este notebook mide y concluye. Adoptar otra estrategia (o, por ejemplo,
  añadir el título del documento al texto de cada chunk por encabezados, que es una hipótesis que sugieren Q19 y Q20)
  sería otra historia y exige la evaluación automática de F2 (Constitución V, ADR-0016).

## 9. Reproducibilidad

* **Tamaño fijo y encabezados son deterministas**: mismo corpus y mismos parámetros dan exactamente los mismos chunks.
* **Los embeddings vienen de Ollama.** Para un mismo texto y modelo son estables en una misma máquina, pero no se garantiza
  que coincidan bit a bit entre máquinas o versiones de Ollama. Por eso el archivo de resultados **no guarda las
  puntuaciones decimales**, sino las **posiciones** (1 a `top_k` o «no encontrado»): eso es lo que debe coincidir entre
  ejecuciones, y con ello el hit rate y el MRR.
* **Tolerancia declarada:** la misma máquina, con la caché de embeddings vacía, debe dar posiciones y métricas
  idénticas. Esta ejecución se contrastó así el 2026-11-10: dos ejecuciones, una de ellas en frío (33 segundos con
  Ollama ya levantado y sin GPU dedicada), con el mismo resultado. En otra máquina, si alguna posición de la estrategia
  semántica cambia por diferencias numéricas, compara primero el `digest` del modelo (abajo y en el JSON): si es
  distinto, no son comparables.

Para repetirlo: borra `notebooks/.cache`, ejecuta el notebook y compara `summary` y `rows` de
`notebooks/results/chunking-results.json` con la versión anterior (ignorando `generated_at`).

In [17]:
print(f"Modelo de embeddings: {config.embed_model} · digest {model_digest or 'no informado'}")
print(f"Corpus: {len(documents)} documentos · preguntas: {len(questions)} · estrategias: {len(Strategy)}")

Modelo de embeddings: bge-m3 · digest 7907646426070047a77226ac3e684fbbe8410524f7b4a74d02837e43f2146bab
Corpus: 24 documentos · preguntas: 20 · estrategias: 3


## Resultados guardados

Los resultados se escriben en `notebooks/results/chunking-results.json` y se imprime su huella `sha256`. Una prueba
automática comprueba que este notebook y ese archivo siguen siendo coherentes.

In [18]:
from chunking_lab.report import results_digest, write_results

write_results(results, config.results_path)
print(f"Resultados guardados en notebooks/results/{config.results_path.name}")
print(f"sha256: {results_digest(config.results_path)}")

Resultados guardados en notebooks/results/chunking-results.json
sha256: b8130dad167c261a6789c5e21d7093e68161855c6b71a3c5db2b4fad9a77c51f
